# Lesson 14: choosing a method and deciding with its output

1. Which method fits which question?
2. Which measure judges a model for its decision?
3. When does a probability become an action?
4. How do groups and components inform a design?

Three data sets: daily US Treasury yields at 11 maturities, 9 February 2006 to 30 September 2026; the monthly term spread, the 10-year Treasury yield minus the 3-month bill rate, with the NBER recession months, April 1953 to September 2026; the policy rates, inflation rates and policy-rate changes of 38 economies in 2025. A pension fund moves part of its equities into bonds when it expects a recession month 12 months later. A false alarm costs it 6 and a missed recession 24 per 100 francs moved; both costs are stylised.

Treasury yields and recession dates: Federal Reserve Board H.15 and NBER via FRED, Federal Reserve Bank of St. Louis. Policy rates and consumer prices: BIS Data Portal. Snapshot 30 September 2026.

In [ ]:
import pathlib

import pandas as pd

DATA_URL = "https://digital-ai-finance.github.io/data-science-msc/data/"
SNAPSHOT = "2026-09-30"


def load(name):
    """Read a snapshot: "treasury-curve-daily", "us-recession-monthly" or "bis-economies"."""
    local = pathlib.Path("../data/market") / f"{SNAPSHOT}-{name}.csv"
    if local.exists():  # run from material/notebooks/ before the files are published
        return pd.read_csv(local)
    url = f"{DATA_URL}{SNAPSHOT}-{name}.csv"
    try:
        return pd.read_csv(url)
    except Exception as err:
        raise RuntimeError(f"{url} not reachable ({err}). Download the file in a browser, upload "
                           "it in the Colab file panel and read it with pd.read_csv.") from err

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

RANDOM_STATE = 42
COST_FA, COST_MISS = 6, 24   # example costs per 100 francs moved: a false alarm, a missed recession
P_STAR = COST_FA / (COST_FA + COST_MISS)

## 1. Which method fits which question?

Two facts set the problem class: whether a question has a target column, and whether the target is a number or a category. A fact about the data then picks the method: linear regression for a number, logistic regression or kNN for a category, k-means for groups, PCA for fewer axes.

In [ ]:
def problem_class(df, target=None):
    """Return the problem class of a question from its target column."""
    if target is None:
        return "no target: groups (k-means) or axes (PCA)"
    if df[target].dropna().nunique() == 2:
        return "classification: logistic regression or kNN"
    if pd.api.types.is_numeric_dtype(df[target]):
        return "regression: linear regression"
    return "classification with more than two classes"


curve = load("treasury-curve-daily").set_index("date")
months = load("us-recession-monthly")
economies = load("bis-economies")
changes = curve.diff().dropna() * 100          # daily changes in basis points
print("10-year yield change:", problem_class(changes, "DGS10"))
print("a recession month 12 months later:", problem_class(months, "recession_in_12m"))
print(f"{len(economies)} economies:", problem_class(economies))

## 2. Yield moves: linear regression and RMSE

The daily change of the 10-year yield on the daily change of the 2-year yield, in basis points (1 bp = 0.01 percentage points). The split is by date: the last 20 per cent of the days are the test days, and every training day lies before every test day.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

cut = int(len(changes) * 0.8)                   # by date: the test days follow the training days
train, test = changes.iloc[:cut], changes.iloc[cut:]
assert train.index[-1] < test.index[0]
lin = LinearRegression().fit(train[["DGS2"]], train["DGS10"])
print(f"10-year change = {lin.intercept_:.2f} + {lin.coef_[0]:.3f} x 2-year change, in bp")


def rmse(df):
    return mean_squared_error(df["DGS10"], lin.predict(df[["DGS2"]])) ** 0.5


# the first training day has no change: 4,131 daily changes over 4,132 training days
print(f"RMSE on {len(train):,} daily changes over {len(train) + 1:,} training days: "
      f"{rmse(train):.2f} bp")
print(f"RMSE on {len(test):,} test days: {rmse(test):.2f} bp")
base = ((test["DGS10"] - train["DGS10"].mean()) ** 2).mean() ** 0.5
print(f"RMSE of the training mean on the test days: {base:.2f} bp")
plt.scatter(test["DGS10"], lin.predict(test[["DGS2"]]), s=6)
plt.plot([-55, 25], [-55, 25], color="grey")
plt.xlabel("actual 10-year change (bp)"); plt.ylabel("predicted 10-year change (bp)")
plt.title(f"{len(test):,} test days from {test.index[0]}"); plt.show()

**Exercise 1.** Predict the 10-year change from the 5-year change (`DGS5`), with the same split by date. Does the test RMSE fall below the 3.92 bp of the 2-year change?

## 3. Recession warnings: probability, threshold, cost

Logistic regression on the spread alone, fitted on the 561 training months to December 1999 and judged on the 297 test months from January 2000 to September 2024. A month gets a warning when its probability reaches the threshold. Moving into bonds costs $\text{alarm} \cdot (1 - p)$ in expectation, with alarm = 6 the cost of a false alarm; staying costs $\text{miss} \cdot p$, with miss = 24 the cost of a missed recession. Both are equal at $p^* = \text{alarm} / (\text{alarm} + \text{miss}) = 0.2$. The cheapest threshold of a sweep is chosen on the training months and judged on the test months.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score

train_m = months[months["split"] == "train"]
test_m = months[months["split"] == "test"]
assert train_m["month"].max() < test_m["month"].min()
logit = LogisticRegression(penalty=None, solver="newton-cholesky", max_iter=1000).fit(
    train_m[["spread"]], train_m["recession_in_12m"].astype(int))
p = logit.predict_proba(test_m[["spread"]])[:, 1]
y = test_m["recession_in_12m"].astype(int).to_numpy()
print(f"intercept {logit.intercept_[0]:.3f}, spread weight {logit.coef_[0, 0]:.3f}; "
      f"test AUC {roc_auc_score(y, p):.3f}; {y.sum()} of {len(y)} test months followed by a recession month 12 months later")
for t in (0.5, P_STAR):
    tn, fp, fn, tp = confusion_matrix(y, p >= t, labels=[0, 1]).ravel()
    precision = tp / (tp + fp) if tp + fp else 0.0
    print(f"threshold {t:.2f}: TP {tp}, FP {fp}, FN {fn}, TN {tn}; "
          f"precision {precision:.3f}, recall {tp / (tp + fn):.3f}")

In [ ]:
p_train = logit.predict_proba(train_m[["spread"]])[:, 1]
y_train = train_m["recession_in_12m"].astype(int).to_numpy()


def cost(truth, prob, t):
    """Cost of the warnings at threshold t: COST_FA per false alarm, COST_MISS per missed recession."""
    tn, fp, fn, tp = confusion_matrix(truth, prob >= t, labels=[0, 1]).ravel()
    return COST_FA * fp + COST_MISS * fn


thresholds = np.round(np.arange(0.01, 0.96, 0.01), 2)
train_costs = [cost(y_train, p_train, t) for t in thresholds]
test_costs = [cost(y, p, t) for t in thresholds]
best = thresholds[int(np.argmin(train_costs))]   # chosen on the training months
cost_at = dict(zip(thresholds, test_costs))
print(f"threshold cheapest on the {len(y_train)} training months: {best:.2f}, "
      f"training cost {min(train_costs):,}")
print(f"test cost at 0.50: {cost_at[0.5]}; at p* = {P_STAR:.2f}: {cost_at[0.2]}; "
      f"at {best:.2f}: {cost_at[best]}; never moving: {COST_MISS * y.sum()}")
plt.plot(thresholds, test_costs, label="cost of the warnings")
plt.axhline(COST_MISS * y.sum(), color="grey", linestyle=":", label="never move")
plt.axvline(P_STAR, color="orange", label=f"p* = {P_STAR:.2f}")
plt.axvline(best, color="grey", linestyle="--", label=f"chosen on the training months: {best:.2f}")
plt.xlabel("threshold"); plt.ylabel(f"cost over {len(y)} test months"); plt.legend(); plt.show()

In [ ]:
future = months[months["split"] == "unlabelled"]
p_future = logit.predict_proba(future[["spread"]])[:, 1]
month, spread, p_last = future["month"].iloc[-1], future["spread"].iloc[-1], p_future[-1]
p_shown = round(p_last, 3)                      # the probability as the slide prints it
print(f"{month}: spread {spread:.2f}, probability {p_shown:.3f}")
print(f"expected cost of moving {(1 - p_shown) * COST_FA:.2f}, of staying {p_shown * COST_MISS:.2f}")
print("move into bonds" if p_last >= P_STAR else "stay invested")
print(f"{(p_future >= P_STAR).sum()} of the {len(future)} months from {future['month'].iloc[0]} "
      f"lie above p*")
plt.plot(future["month"], p_future, marker="o")
plt.axhline(P_STAR, color="orange"); plt.xticks(rotation=90)
plt.ylabel("probability of a recession month 12 months later"); plt.show()

**Exercise 2.** A false alarm costs 5 and a missed recession 45. Where do $p^*$ and the threshold cheapest on the training months lie, what does that threshold cost on the test months, and does the fund move in September 2026?

**Exercise 3.** Fit `KNeighborsClassifier(n_neighbors=101)`, the k that Lesson 11's forward folds chose, on `spread` and `tbill_change_12m`, after a `StandardScaler` fitted on the training months only. Compute its precision and recall at 0.5 and at $p^*$, and set them beside those of the spread model.

## 4. Groups and components

Türkiye's policy rate of 42.83 per cent stays an outlier after standardising, so Türkiye stands apart; the scaler is fitted on the other 37 economies, and k-means runs on them. A centroid reads in the units of its features, here per cent. PCA runs on the unscaled levels of the 11 yields.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

features = ["policy_rate_2025", "cpi_inflation_2025", "policy_change_2021_2025"]
rest = economies["code"] != "TR"               # Türkiye stands apart: policy rate 42.83 per cent
x = StandardScaler().fit_transform(economies.loc[rest, features])
for k in range(2, 7):
    labels = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(x)
    sizes = sorted(np.bincount(labels).tolist(), reverse=True)
    print(f"k = {k}: silhouette {silhouette_score(x, labels):.3f}, sizes {sizes}")
economies["group"] = "Türkiye"
economies.loc[rest, "group"] = KMeans(n_clusters=2, n_init=10,
                                      random_state=RANDOM_STATE).fit_predict(x).astype(str)
for group, members in economies.groupby("group"):
    name = "Türkiye" if group == "Türkiye" else f"{len(members)} economies"
    print(f"{name}: policy rate {members['policy_rate_2025'].mean():.2f}, "
          f"inflation {members['cpi_inflation_2025'].mean():.2f}, "
          f"change {members['policy_change_2021_2025'].mean():.2f}; {', '.join(members['code'])}")

In [ ]:
from sklearn.decomposition import PCA

pca = PCA().fit(curve)
share = 100 * pca.explained_variance_ratio_
print("variance shares: " + ", ".join(f"{s:.1f}" for s in share[:4]) + " per cent; "
      f"two components {share[:2].sum():.1f} per cent")
xy = pca.transform(curve)[:, :2]
year = pd.to_datetime(curve.index).year
plt.scatter(xy[:, 0], xy[:, 1], c=year, s=3, cmap="viridis")
plt.colorbar(label="year")
plt.xlabel("component 1: level"); plt.ylabel("component 2: slope")
plt.title(f"{share[:2].sum():.1f} per cent of the curve's variance on {len(curve):,} days"); plt.show()

**Exercise 4.** Put Türkiye back: fit the scaler on all 38 economies and rerun the silhouettes for k = 2 to 6. Which k wins, and which economy sits alone?

Then fill the question table for the three questions, 10-year yield move, a recession month 12 months later and groups of economies: problem class, method, measure, and the decision each one serves.

## Newer data (optional)

FRED serves the live series. Their numbers differ from the snapshot of 30 September 2026 and from the slides.

In [ ]:
FRED = "https://fred.stlouisfed.org/graph/fredgraph.csv?id="
BIS = "https://stats.bis.org/api/v1/data/{}?startPeriod={}&format=csv"


def fred(*ids):
    """Live FRED series, e.g. fred("GS10", "TB3MS", "USREC"); "." marks a missing value."""
    return pd.read_csv(FRED + ",".join(ids), na_values=["."], parse_dates=["observation_date"])


def bis(flow, start):
    """Live BIS series, e.g. bis("WS_CBPOL/M..", "2021-12")."""
    return pd.read_csv(BIS.format(flow, start))


live = fred("GS10", "TB3MS").dropna()
spread_now = live["GS10"].iloc[-1] - live["TB3MS"].iloc[-1]
p_now = logit.predict_proba(pd.DataFrame({"spread": [spread_now]}))[0, 1]
print(f"{live['observation_date'].iloc[-1]:%Y-%m}: spread {spread_now:.2f}, probability {p_now:.3f}")